In [11]:

from pathlib import Path
import pandas as pd
import numpy as np
repo_root = Path.cwd().resolve().parents[1]
data_path = 'week2/week2_customer_transactions_messy.csv'
df = pd.read_csv(data_path)
print('Loaded:', data_path)
print('Shape:', df.shape)
df.head()


Loaded: week2/week2_customer_transactions_messy.csv
Shape: (11, 9)


,transaction_id,customer_id,transaction_date,amount,currency,payment_method,status,region,last_updated
0,T0001,C100,2026-01-05,120.50,EUR,card,completed,DE,2026-01-05
1,T0002,C101,2026/01/06,0.00,EUR,CARD,completed,de,2026-01-20
2,T0003,C102,06-01-2026,-35.00,USD,bank_transfer,completed,US,2026-01-07
3,T0004,NaN,2026-01-07,250.00,EUR,card,pending,FR,2026-01-08
4,T0005,C104,2026-01-07,89.99,EURO,cash,completed,DE,2026-01-09


The dataset appears to contain customer transaction records, including fields such as transaction ID, customer ID, date, product, quantity, and price and it can be used for revenue tracking ,fraud detection and operational reporting 

In [12]:
issue_table = pd.DataFrame([['Missing customer_id','Completeness','Impacts customer analytics'],['Duplicate transaction_id','Uniqueness','May double count revenue']], columns=['Issue','Dimension','Impact'])
issue_table

,Issue,Dimension,Impact
0,Missing customer_id,Completeness,Impacts customer analytics
1,Duplicate transaction_id,Uniqueness,May double count revenue


Some columns such as `customer_id` and `amount` contain missing values.  
Missing values in key identifiers can significantly impact analysis.

In [13]:
kpis={}
kpis['Completeness Rate']=1-(df.isna().sum().sum()/(df.shape[0]*df.shape[1]))
kpis['Duplication Rate']=df.duplicated(subset=['transaction_id']).mean()
amount=pd.to_numeric(df['amount'], errors='coerce')
date_ok=pd.to_datetime(df['transaction_date'], errors='coerce', format='mixed').notna()
kpis['Error Rate']=(amount.isna() | (amount<0) | ~date_ok).mean()
pd.DataFrame({'KPI':list(kpis.keys()), 'Value':list(kpis.values())})


,KPI,Value
0,Completeness Rate,0.959596
1,Duplication Rate,0.090909
2,Error Rate,0.272727


The completeness rate indicates moderate/high missing data. The duplication rate suggests potential data redundancy
 The error rate highlights issues in data validation.

In [15]:
rules={
'transaction_id_required': df['transaction_id'].isna() | (df['transaction_id'].astype(str).str.strip()==''),
'amount_non_negative': pd.to_numeric(df['amount'], errors='coerce')<0,
'transaction_date_valid': pd.to_datetime(df['transaction_date'], errors='coerce', format='mixed').isna(),
}
pd.DataFrame({k:int(v.sum()) for k,v in rules.items()}, index=['affected_rows']).T



,affected_rows
transaction_id_required,0
amount_non_negative,1
transaction_date_valid,1


1. Customer ID must not be null  
2. Transaction amount must be greater than 0  
3. Transaction date must be valid  
4. Transaction ID must be unique  

In [16]:
audit_summary = pd.DataFrame([['Example issue',0,'Medium','Example next action']], columns=['issue_type','affected_rows','severity','recommended_next_action'])
audit_summary



,issue_type,affected_rows,severity,recommended_next_action
0,Example issue,0,Medium,Example next action


cleaning plan - 
- Remove duplicate records  
- Handle missing values 
- Standardize categorical values  
- Convert date columns to proper format  
- Validate numeric ranges  
- Recalculate derived fields  

In [ ]:
import pandas as pd

def summarize_rule_violations(rule_dictionary):
    """
    Summarize affected-row counts for each validation rule.
    Parameters:
    rule_dictionary : dict
        Dictionary where keys are rule names and values are boolean masks.
        Each boolean mask should contain True for rows that violate the rule.

    Returns:
    pd.DataFrame
        A table showing each rule and the number of affected rows.
    """
    return pd.DataFrame({
        "rule_name": list(rule_dictionary.keys()),
        "affected_rows": [int(mask.sum()) for mask in rule_dictionary.values()]
    }).sort_values("affected_rows", ascending=False)

In [18]:
rules = {
    "Missing customer_id": df["customer_id"].isna(),
    "Missing transaction_id": df["transaction_id"].isna(),
    "Duplicate transaction_id": df.duplicated(subset=["transaction_id"]),
    "Invalid transaction amount": df["amount"] <= 0,
    "Invalid transaction date": pd.to_datetime(df["transaction_date"], errors="coerce").isna()
}

In [19]:
rule_summary = summarize_rule_violations(rules)
rule_summary

,rule_name,affected_rows
4,Invalid transaction date,3
3,Invalid transaction amount,2
0,Missing customer_id,1
2,Duplicate transaction_id,1
1,Missing transaction_id,0


- `rule_dictionary` is the input parameter. It stores validation rule names as keys and boolean masks as values.
- Each boolean mask identifies problematic rows, where `True` means the row violates that rule.
- I did not use a default value because the function should only run when specific validation rules are provided.
- Changing the rules in `rule_dictionary` changes the affected row counts.
- These affected row counts can influence audit KPIs such as error rate and overall data quality assessment.